# Cleaning of geoapify json response

In [5]:
import json
from pathlib import Path
import pandas as pd

In [13]:
RAW_FILE = "../raw/venues_raw.json"
CLEAN_JSON = "../cleaned/venues_cleaned.json"
CLEAN_CSV = "../cleaned/venues_cleaned.csv"

with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

print(f"Raw records loaded: {len(raw_data)}")

df = pd.json_normalize(raw_data)
print(df.shape)
print(df.columns)

Raw records loaded: 400
(400, 392)
Index(['type', 'properties.name', 'properties.country',
       'properties.country_code', 'properties.state', 'properties.city',
       'properties.postcode', 'properties.suburb', 'properties.quarter',
       'properties.street',
       ...
       'properties.datasource.raw.min_height',
       'properties.datasource.raw.payment:cards',
       'properties.datasource.raw.roof:material',
       'properties.payment_options.cards', 'properties.datasource.raw.leisure',
       'properties.datasource.raw.max_age', 'properties.restrictions.max_age',
       'properties.datasource.raw.barrier',
       'properties.datasource.raw.fence_type',
       'properties.datasource.raw.surface'],
      dtype='str', length=392)


### The below code is to save the starting count

In [14]:
len(df)

400

### Save the fields that we actually need

In [16]:
columns2 = [
    "properties.place_id",
    "properties.name",
    "properties.categories",
    "properties.formatted",
    "properties.postcode",
    "properties.district",
    "properties.lat",
    "properties.lon",
    "properties.website",
    "properties.opening_hours"
]

df = df.reindex(columns=columns2) # Makes the dataframes columns match the order and names in the columns2 list

### Rename the columns to match the database fields:

In [17]:
df = df.rename(columns={
    "properties.place_id": "geoapify_place_id",
    "properties.name": "name",
    "properties.categories": "categories",
    "properties.formatted": "address",
    "properties.postcode": "postcode",
    "properties.district": "borough",
    "properties.lat": "latitude",
    "properties.lon": "longitude",
    "properties.website": "website",
    "properties.opening_hours": "opening_hours"
})

### Clean the category field to match the database

- Database schema only has one category field wheras API returns multiple.

In [ ]:
def get_category(categories):
    if isinstance(categories, list) and len(categories) > 0:
        return categories[-1]

    return None

df["category"] = df["categories"].apply(get_category)

df = df.drop(columns=["categories"])